## Colab Setup

Run this first in Colab to install the OCR, KIE, and evaluation dependencies before executing the notebook cells below. After it finishes, restart the runtime before running the pipeline cells.

In [ ]:
# Colab dependency installation for OCR + KIE pipeline
# Run this cell once at the top of a fresh Colab runtime.
# After it finishes, restart the runtime before running the notebook cells below.
%pip install -q --upgrade pip
%pip install -q numpy==1.26.4 pypdfium2==4.30.0 pdf2image==1.17.0 opencv-python-headless==4.11.0.86 pillow==11.3.0 surya-ocr==0.17.1 transformers==4.56.1 torch==2.7.1 img2table==1.4.2 pandas==2.3.3 spacy==3.7.5 regex==2026.4.4
%pip install -q https://github.com/explosion/spacy-models/releases/download/fr_core_news_lg-3.7.0/fr_core_news_lg-3.7.0-py3-none-any.whl
print("Colab dependencies installed. Restart the runtime now, then run the notebook cells.")

# Complete IDP Pipeline: OCR + KIE + Evaluation

**Integrated Jupyter notebook for document processing, key information extraction, and evaluation.**

This notebook combines:
- **OCR Module**: Document preprocessing, layout detection, text recognition, table extraction
- **KIE Module**: Key information extraction (sender, receiver, date, reference, etc.)
- **Evaluation**: Comprehensive metrics (CER, WER, F1, IoU) with CSV export

Run cells sequentially to process documents and generate evaluation reports.

In [ ]:
# ============================================================================
# SECTION 1: Import Required Libraries and Setup Paths
# ============================================================================

import multiprocessing
multiprocessing.freeze_support()

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"

import csv
import json
import re
import sys
import time
import tempfile
from collections import Counter, defaultdict
from pathlib import Path
from typing import Optional, Union

import cv2
import numpy as np
from PIL import Image

try:
    import pypdfium2 as pdfium
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pypdfium2"])
    import pypdfium2 as pdfium

# Setup paths
ROOT = Path.cwd()
DOCS_DIR = ROOT / "documents"
GT_CSV = DOCS_DIR / "generated_documents.csv"

print(f"Working directory: {ROOT}")
print(f"Documents directory: {DOCS_DIR}")
print(f"Ground truth CSV: {GT_CSV}")

## 1. OCR Preprocessor Functions

These functions handle document loading, noise reduction, and image preprocessing for optimal OCR.

In [ ]:
# Load document from PDF or image
def load_document(file_path: str) -> list[np.ndarray]:
    """Load document (PDF or image) and return list of page images."""
    try:
        import pypdfium2 as pdfium
    except ImportError:
        raise ImportError("pypdfium2 required. Install: pip install pypdfium2")
    
    path = Path(file_path)
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        pdf = pdfium.PdfDocument(str(path))
        pages = []
        for i in range(len(pdf)):
            bitmap = pdf[i].render(scale=300/72)
            pil_image = bitmap.to_pil()
            pages.append(cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR))
        return pages
    elif suffix in [".jpg", ".jpeg", ".png"]:
        img = cv2.imread(str(path))
        if img is None:
            raise ValueError(f"Could not read image: {file_path}")
        return [img]
    else:
        raise ValueError(f"Unsupported format: {suffix}")


def _is_digital(image: np.ndarray) -> bool:
    """Detect if image is clean digital render vs. degraded scan."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    mean = float(np.mean(gray))
    std = float(np.std(gray))
    return mean > 200 and std < 60


def _sharpen(image: np.ndarray) -> np.ndarray:
    """Apply mild unsharp mask for digital documents."""
    blurred = cv2.GaussianBlur(image, (0, 0), 1.5)
    return cv2.addWeighted(image, 1.5, blurred, -0.5, 0)


def deskew(image: np.ndarray) -> np.ndarray:
    """Correct image skew/rotation."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.bitwise_not(gray)
    coords = np.column_stack(np.where(gray > 0))
    if len(coords) == 0:
        return image
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = 90 + angle
    elif angle > 45:
        angle = angle - 90
    if abs(angle) < 0.5:
        return image
    h, w = image.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


def denoise(image: np.ndarray) -> np.ndarray:
    """Remove noise from image."""
    return cv2.fastNlMeansDenoisingColored(image, None, 10, 10, 7, 21)


def binarize(image: np.ndarray) -> np.ndarray:
    """Convert to binary (black/white) for better OCR."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 11)
    return cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)


def preprocess(image: np.ndarray) -> np.ndarray:
    """Preprocess image: adapt strategy based on image type."""
    if _is_digital(image):
        return _sharpen(image)
    image = deskew(image)
    image = denoise(image)
    image = binarize(image)
    return image


def detect_circular_stamps(image: np.ndarray) -> list[tuple]:
    """Detect circular stamps/watermarks in image."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                               param1=50, param2=30, minRadius=30, maxRadius=200)
    if circles is None:
        return []
    circles = np.round(circles[0, :]).astype("int")
    return [(x, y, r) for x, y, r in circles]

print("✓ OCR preprocessing functions loaded")

## 2. OCR Model Functions (Surya - Layout, Text, Tables)

In [ ]:
# Lazy-load layout detection models
_layout_predictors = None

def _load_layout_predictors():
    """Lazy-load Surya OCR models."""
    global _layout_predictors
    if _layout_predictors is None:
        try:
            from surya.detection import DetectionPredictor
            from surya.layout import LayoutPredictor
            from surya.recognition import RecognitionPredictor
            from surya.foundation import FoundationPredictor
            from surya.settings import settings

            try:
                from surya.table_rec import TableRecPredictor
            except Exception:
                TableRecPredictor = None
                print("Warning: surya table recognition unavailable; continuing without it.")

            _layout_predictors = {
                "detection": DetectionPredictor(),
                "recognition": RecognitionPredictor(
                    FoundationPredictor(checkpoint=settings.RECOGNITION_MODEL_CHECKPOINT)
                ),
                "layout": LayoutPredictor(
                    FoundationPredictor(checkpoint=settings.LAYOUT_MODEL_CHECKPOINT)
                ),
            }
            if TableRecPredictor is not None:
                _layout_predictors["table_rec"] = TableRecPredictor()
        except ImportError:
            raise ImportError("surya required. Install: pip install surya-ocr")

def get_predictors():
    """Get cached OCR/layout predictor instances."""
    _load_layout_predictors()
    return _layout_predictors

def ocr_full_page(pil_image):
    """Run full-page OCR with Surya."""
    from PIL import Image
    predictors = get_predictors()
    results = predictors["recognition"](
        [pil_image],
        det_predictor=predictors["detection"],
        task_names=None,
        sort_lines=True
    )
    result = results[0]
    if not result.text_lines:
        return "", 0.0
    text = "\n".join([line.text for line in result.text_lines])
    avg_conf = float(np.mean([line.confidence for line in result.text_lines]))
    return text, avg_conf

def _run_ocr(pil_image):
    """Execute OCR on a PIL image."""
    predictors = get_predictors()
    results = predictors["recognition"](
        [pil_image],
        det_predictor=predictors["detection"],
        task_names=None,
        sort_lines=True,
    )
    return results[0]

def _build_text(ocr_result):
    """Extract text and confidence from OCR result."""
    lines = getattr(ocr_result, "text_lines", [])
    if not lines:
        return "", 0.0
    text = "\n".join(l.text for l in lines if l.text and l.text.strip())
    conf = float(np.mean([l.confidence for l in lines]))
    return text, conf

# Table extraction with img2table
def _extract_with_img2table(img, page_index=0):
    """Extract tables using img2table library."""
    try:
        from img2table.document import Image as Img2TableImage
        import polars as pl
    except ImportError:
        return [], []

    preprocessed = _preprocess_for_table_detection(img)
    page_h, page_w = img.shape[:2]
    page_area = page_h * page_w

    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        tmp_path = tmp.name
        cv2.imwrite(tmp_path, preprocessed)

    try:
        doc = Img2TableImage(src=tmp_path, detect_rotation=False)
        tables = doc.extract_tables(
            implicit_rows=True,
            implicit_columns=False,
            borderless_tables=False,
            min_confidence=40
        )
        if not tables:
            return [], []

        filtered_tables, bboxes = [], []
        for table in tables:
            if hasattr(table, 'bbox') and table.bbox is not None:
                bbox = (table.bbox.x1, table.bbox.y1, table.bbox.x2, table.bbox.y2)
            else:
                bbox = (0, 0, 0, 0)
            x1, y1, x2, y2 = bbox
            table_area = max(0, x2 - x1) * max(0, y2 - y1)
            if table_area / page_area > 0.30:
                continue
            filtered_tables.append(table)
            bboxes.append(bbox)

        return filtered_tables, bboxes
    finally:
        os.unlink(tmp_path)


def _preprocess_for_table_detection(img):
    """Preprocess image for robust table detection."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=5, templateWindowSize=7, searchWindowSize=15)
    return cv2.cvtColor(denoised, cv2.COLOR_GRAY2BGR)


def _convert_img2table_results(tables, bboxes):
    """Convert img2table results to standardized format."""
    results = []
    for i, (table, bbox) in enumerate(zip(tables, bboxes)):
        df = table.df.fillna("")
        rows = df.values.tolist()
        total = df.size
        filled = sum(1 for row in rows for cell in row if str(cell).strip())
        confidence = round(filled / total, 2) if total > 0 else 0.0
        n_rows = len(rows)
        n_cols = len(rows[0]) if rows else 0
        results.append({
            "table_index": i,
            "data": rows,
            "bbox": [int(v) for v in bbox],
            "confidence": confidence,
            "method": "img2table",
            "shape": [n_rows, n_cols]
        })
    return results

def extract_tables(img, ocr_result=None, page_index=0):
    """Extract tables from image using img2table."""
    tables, bboxes = _extract_with_img2table(img, page_index)
    results = _convert_img2table_results(tables, bboxes)
    return results, bboxes

def mask_table_regions_from_text(ocr_result, table_bboxes):
    """Mask out table regions from OCR text lines."""
    if not hasattr(ocr_result, 'text_lines'):
        return ocr_result
    filtered_lines = []
    for line in ocr_result.text_lines:
        if not line.polygon:
            filtered_lines.append(line)
            continue
        xs = [p[0] for p in line.polygon]
        ys = [p[1] for p in line.polygon]
        line_x1, line_y1 = min(xs), min(ys)
        line_x2, line_y2 = max(xs), max(ys)
        in_table = False
        for bbox in table_bboxes:
            t_x1, t_y1, t_x2, t_y2 = bbox
            if not (line_x2 < t_x1 or line_x1 > t_x2 or line_y2 < t_y1 or line_y1 > t_y2):
                in_table = True
                break
        if not in_table:
            filtered_lines.append(line)
    ocr_result.text_lines = filtered_lines
    return ocr_result

def rebuild_body_text(filtered_lines):
    """Rebuild body text from filtered OCR lines."""
    if not filtered_lines:
        return ""
    return "\n".join(line.text for line in filtered_lines if line.text and line.text.strip())

def build_page_output(page_number, full_text, text_confidence, tables=None):
    """Build structured output for a single OCR page."""
    return {
        "page": page_number,
        "raw_text": re.sub(r'<[^>]+>', '', full_text).strip(),
        "confidence": round(text_confidence, 2),
        "tables": tables or []
    }

print("✓ OCR model functions and table extraction loaded")

## 3. KIE Field Extraction Functions

Extract sender, receiver, date, reference, subject, attachments, and body from OCR text using regex patterns tailored for French administrative documents.

In [ ]:
# Field extraction helper functions
def _norm_ocr_label(line):
    """Normalize OCR labels."""
    s = (line or "").replace("'", "'").strip()
    s = re.sub(r"[|¦]+", ":", s)
    s = re.sub(r"^\s*R[EF]{1,2}\s*[:\-]?\s*", "REF: ", s, flags=re.IGNORECASE)
    s = re.sub(r"^\s*Réf\s*[:\-]?\s*", "Réf : ", s, flags=re.IGNORECASE)
    s = re.sub(r"^\s*Obje[tl1]\s*[:\-]?\s*", "Objet : ", s, flags=re.IGNORECASE)
    s = re.sub(r"^\s*P[\.\s]*[Jj1][\.\s]*[:\-]?\s*", "P.J : ", s, flags=re.IGNORECASE)
    s = re.sub(r"\s{2,}", " ", s)
    return s

def _normalized_lines(raw_text):
    """Split and normalize text lines."""
    return [_norm_ocr_label(l) for l in raw_text.split("\n")]

# Regex patterns for French administrative documents
_REF_CODE = (
    r"[A-Z]{2,}(?:[/-][A-Z0-9]{2,})*"
    r"(?:"
    r"[/-]N[°º]\s*\d*[/-]\d{4}"
    r"|[/-]N[°º]\s*\d+"
    r"|[/-]\d{4}[/-]\d+"
    r"|[/-]\d{4}"
    r")"
)

_REF_BODY_RE = re.compile(r"(?:Réf|REF)\s*:\s*(" + _REF_CODE + r"(?:\s+du\s+[\d/]+)?)", re.IGNORECASE)
_OBJET_RE = re.compile(r"Objet\s*:\s*(.+?)(?:\n|$)", re.IGNORECASE)
_PJ_RE = re.compile(r"(?:P\.?\s*J\.?\s*:?\s*|Pi[eè]ces?\s+jointes?\s*:?\s*)(.+?)(?:\n|$)", re.IGNORECASE)

_BODY_START_RE = re.compile(
    r"(?:^|\n)(?:Monsieur\s*[;:,]|"
    r"Faisant\s+suite|Dans\s+le\s+cadre|Suite\s+\u00e0|"
    r"J['\u2019]ai\s+l['\u2019]honneur|Je\s+vous\s+prie|"
    r"Nous\s+avons\s+l['\u2019]honneur|Conform[eé]ment|"
    r"Suite\s+aux|Suite\s+à|"
    r"Nous\s+avons\s+constat[eé]|Nous\s+vous\s+sollicitons|"
    r"Nous\s+vous\s+informons|Nous\s+vous\s+transmettons|"
    r"Le\s+D[eé]partement|Nous\s+avons\s+proc[eé]d[eé]|"
    r"Suite\s+aux\s+r[eé]centes|Nous\s+vous\s+adressons)\s*\n?",
    re.IGNORECASE | re.MULTILINE
)

_BODY_END_RE = re.compile(
    r"(?:Veuillez\s+agréer|Veuillez\s+recevoir|Veuillez\s+trouver|"
    r"Recevez|Je\s+vous\s+prie\s+d['\u2019]agréer|"
    r"Dans\s+l.attente|Comptant\s+sur|En\s+vous\s+remerciant|"
    r"Copie\s+[aà]\s*:|Direction\s+G[eé]n[eé]rale\s*,\s*Service|"
    r"Direction\s+des\s+Op[eé]rations\s*$|"
    r"D[eé]partement\s+Risques\s*,)",
    re.IGNORECASE
)

_ARABIC_RE = re.compile(r"[\u0600-\u06FF]")
_URL_RE = re.compile(r"(?:www\.|http|\.dz|\.com|\.org)", re.IGNORECASE)
_NOISE_RE = re.compile(r"^[^A-Za-zÀ-ÿ]{0,3}[^A-Za-zÀ-ÿ\s]{3,}")
_FOOTER_RE = re.compile(r"(?:Quartier|Tél\s*:|Bab\s+Ezzouar|BP\s*\d)", re.IGNORECASE)

def _is_noise(line):
    """Detect OCR artifacts and noise."""
    if re.search(r"(?:\+{1,}|%{1,}|N8X|oloX|米)", line):
        return True
    if _ARABIC_RE.search(line):
        return True
    if _URL_RE.search(line):
        return True
    if _NOISE_RE.search(line):
        return True
    if _FOOTER_RE.search(line):
        return True
    if sum(c.isalpha() for c in line) / max(len(line), 1) < 0.4:
        return True
    if len(line) < 3:
        return True
    return False

def _extract_sender(raw_text):
    """Extract sender field from document."""
    lines = [l.strip() for l in _normalized_lines(raw_text) if l.strip()]
    for line in lines[:12]:
        m = re.match(r"De\s*:\s*(.+)", line, re.IGNORECASE)
        if m:
            val = m.group(1).strip()
            if not _is_noise(val):
                return val
    for line in lines[:8]:
        if re.search(r"\b(REF|Réf|Objet|P\.J)\b", line, re.IGNORECASE):
            break
        if _is_noise(line):
            continue
        return line
    return None

def _extract_ref_header(raw_text):
    """Extract reference number from document header."""
    lines = _normalized_lines(raw_text)
    first_lines = [l.strip() for l in lines[:25]]
    for i, line in enumerate(first_lines[:-1]):
        if re.match(r"^REF\s*:?\s*$", line, re.IGNORECASE):
            m = re.search(_REF_CODE, first_lines[i + 1])
            if m:
                return " ".join(m.group(0).split())
    for line in first_lines:
        m = re.search(r"REF\s*:\s*(" + _REF_CODE + r")", line, re.IGNORECASE)
        if m:
            return " ".join(m.group(1).split())
    return None

def _extract_date(raw_text):
    """Extract date field."""
    norm_text = "\n".join(_normalized_lines(raw_text))
    m = re.search(r"Alger\s*:?\s*,?\s*(?:le\s*)?\n\s*(\d{2}[/\-]\d{2}[/\-]\d{4})", norm_text, re.IGNORECASE)
    if m:
        return m.group(1)
    for line in norm_text.split("\n")[:25]:
        m = re.search(r"\b(\d{2}[/\-]\d{2}[/\-]\d{4})\b", line)
        if m:
            return m.group(1)
    return None

def _extract_receiver(raw_text):
    """Extract receiver/recipient field."""
    lines = [l.strip() for l in _normalized_lines(raw_text)]
    norm_text = "\n".join(lines)
    m = re.search(r"(?im)^\s*[AÀ]\s*(?:l['\u2019]attention\s+(?:de|du|des)|:)\s*(.+?)\s*$", norm_text)
    if m:
        return m.group(1).strip()
    return None

def _extract_objet(raw_text):
    """Extract subject/object field."""
    norm_text = "\n".join(_normalized_lines(raw_text))
    lines = [l.strip() for l in norm_text.split("\n")]
    for i, line in enumerate(lines):
        m = re.match(r"^\s*Objet\s*:?\s*(.*)$", line, re.IGNORECASE)
        if m:
            first_line = (m.group(1) or "").strip()
            if not first_line:
                for nxt in lines[i+1:i+4]:
                    if nxt.strip():
                        first_line = nxt.strip()
                        break
            return first_line if first_line else None
    return None

def _extract_pj(raw_text):
    """Extract attached pieces (Pièces Jointes) field."""
    norm_text = "\n".join(_normalized_lines(raw_text))
    lines = [l.strip() for l in norm_text.split("\n")]
    for i, line in enumerate(lines):
        m = re.match(r"^\s*P\.?\s*J\.?\s*:?\s*(.*)$", line, re.IGNORECASE)
        if m:
            return (m.group(1) or "").strip()
    return None

def _extract_body(raw_text):
    """Extract document body text."""
    norm_text = "\n".join(_normalized_lines(raw_text))
    start_m = _BODY_START_RE.search(norm_text)
    if not start_m:
        return None
    body_start = norm_text[start_m.start():]
    end_m = _BODY_END_RE.search(body_start)
    if end_m:
        body_text = body_start[:end_m.start()].strip()
    else:
        body_text = body_start.strip()
    return body_text if body_text else None

def extract_fields(raw_text):
    """Extract all KIE fields from OCR text."""
    return {
        "sender": _extract_sender(raw_text),
        "receiver": _extract_receiver(raw_text),
        "date": _extract_date(raw_text),
        "ref_header": _extract_ref_header(raw_text),
        "objet": _extract_objet(raw_text),
        "pj": _extract_pj(raw_text),
        "body": _extract_body(raw_text),
    }

# Document type detection
_SUBTYPE_PATTERNS = [
    ("demande", [r"\bdemande\b", r"\bsollicite\b", r"\bprie\b"]),
    ("transmission", [r"\btransmet[s]?\b", r"\bci[- ]joint\b"]),
    ("information", [r"\binforme[r]?\b", r"\bnotif\w+\b"]),
]

def detect_doc_type(raw_text):
    """Detect document type and subtype."""
    text_lower = raw_text.lower()
    subtype = "autre"
    for name, patterns in _SUBTYPE_PATTERNS:
        for pattern in patterns:
            if re.search(pattern, text_lower):
                subtype = name
                break
        if subtype != "autre":
            break
    return {
        "doc_type": "lettre_administrative",
        "doc_subtype": subtype
    }

# Output builders
def build_document_output(doc_id, pages_ocr, pages_fields, doc_type):
    """Build structured KIE output."""
    pages = []
    for ocr, fields in zip(pages_ocr, pages_fields):
        pages.append({
            "page_number": ocr.get("page", 1),
            "ocr_confidence": ocr.get("confidence", 0.0),
            "fields": fields,
            "tables": ocr.get("tables", [])
        })
    return {
        "document_id": doc_id,
        "total_pages": len(pages),
        "doc_type": doc_type.get("doc_type"),
        "doc_subtype": doc_type.get("doc_subtype"),
        "pages": pages
    }

print("✓ KIE field extraction functions loaded")

## 4. Evaluation Metrics

Compute Character Error Rate (CER), Word Error Rate (WER), F1 scores, Exact Match, and other metrics for assessing OCR and KIE quality.

In [ ]:
# Normalize strings for comparison
def _norm(s):
    """Normalize string for comparison."""
    return re.sub(r"\s+", " ", str(s or "").lower()).strip()

def _ref_norm(s):
    """Normalize reference codes."""
    return re.sub(r"[-\s]+", "/", _norm(s)).strip("/")

# Levenshtein distance
def _edit_distance(a, b):
    """Compute Levenshtein distance."""
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            tmp = dp[j]
            dp[j] = prev if a[i-1] == b[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev = tmp
    return dp[n]

# Metric calculations
def calc_cer(hyp, ref):
    """Calculate character error rate."""
    h, r = list(_norm(hyp)), list(_norm(ref))
    if not r:
        return 0.0 if not h else 1.0
    return min(_edit_distance(h, r) / len(r), 1.0)

def calc_wer(hyp, ref):
    """Calculate word error rate."""
    h, r = _norm(hyp).split(), _norm(ref).split()
    if not r:
        return 0.0 if not h else 1.0
    return min(_edit_distance(h, r) / len(r), 1.0)

def calc_levenshtein(hyp, ref):
    """Calculate raw Levenshtein distance (character-level)."""
    return _edit_distance(list(_norm(hyp)), list(_norm(ref)))

def calc_text_accuracy(hyp, ref):
    """Calculate text accuracy: 1 - CER."""
    return max(0.0, 1.0 - calc_cer(hyp, ref))

def calc_exact_match(pred, gold, field=""):
    """Calculate exact match for a field."""
    if not gold:
        return float("nan")
    p = _ref_norm(pred) if "ref" in field else _norm(pred)
    g = _ref_norm(gold) if "ref" in field else _norm(gold)
    return 1.0 if p == g else 0.0

def calc_token_prf(pred, gold):
    """Calculate token-level precision, recall, F1."""
    if not gold:
        return float("nan"), float("nan"), float("nan")
    p_toks = _norm(pred).split()
    g_toks = _norm(gold).split()
    if not p_toks and not g_toks:
        return 1.0, 1.0, 1.0
    if not p_toks or not g_toks:
        return 0.0, 0.0, 0.0
    common = sum((Counter(p_toks) & Counter(g_toks)).values())
    prec = common / len(p_toks)
    rec = common / len(g_toks)
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return prec, rec, f1

def calc_field_accuracy(pred, gold, threshold=0.5):
    """Calculate field-level accuracy."""
    if not gold:
        return float("nan")
    _, _, f1 = calc_token_prf(pred, gold)
    return 1.0 if (f1 == f1 and f1 >= threshold) else 0.0

def calc_iou_text(pred, gold):
    """Calculate text-based IoU as bounding-box quality proxy."""
    if not gold:
        return float("nan")
    p_set = set(_norm(pred).split())
    g_set = set(_norm(gold).split())
    if not p_set and not g_set:
        return 1.0
    inter = len(p_set & g_set)
    union = len(p_set | g_set)
    return inter / union if union > 0 else 0.0

# Helper functions for display
def _safe_mean(vals):
    """Compute mean, ignoring NaN values."""
    clean = [v for v in vals if v == v]
    return sum(clean) / len(clean) if clean else float("nan")

def _fmt(v):
    """Format value for display."""
    if v != v:
        return "n/a"
    if isinstance(v, float):
        return f"{v:.4f}"
    return str(v)

print("✓ Evaluation metrics loaded")

## 5. Load Ground Truth Data

Load reference data from CSV for evaluation.

In [ ]:
# KIE field to CSV column mapping
KIE_FIELDS = {
    "sender":     "Source",
    "receiver":   "Destination",
    "date":       "Date",
    "ref_header": "Ref",
    "objet":      "Objet",
    "pj":         "Pj",
    "body":       "Content",
}

def _load_gt():
    """Load ground truth data from CSV."""
    gt = {}
    if not GT_CSV.exists():
        print(f"Warning: Ground truth CSV not found at {GT_CSV}")
        return gt
    
    with open(GT_CSV, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            m = re.search(r"doc_(\d+)", str(row.get("filename", "")))
            if m:
                gt[m.group(1)] = row
    return gt

def _gt_full_text(row):
    """Concatenate all GT fields into full text."""
    parts = [
        row.get("Source", ""),
        row.get("Destination", ""),
        row.get("Date", ""),
        row.get("Ref", ""),
        row.get("Objet", ""),
        row.get("Pj", ""),
        row.get("Content", ""),
        row.get("Copie", ""),
        row.get("Table", ""),
    ]
    return " ".join(p.strip() for p in parts if p and str(p).strip())

print("✓ Ground truth loading functions ready")

## 6. Main Pipeline Entry Points

**process_document()**: Run OCR on PDF/image  
**extract()**: Extract key information from OCR output

In [ ]:
def process_document(file_path, include_debug=False):
    """
    OCR entry point: process a document (PDF or image) and extract structured data.
    
    Args:
        file_path: Path to document file
        include_debug: If True, return (results, debug_pages); else just results
    
    Returns:
        List of page outputs with OCR text, tables, confidence scores
    """
    raw_pages = load_document(file_path)
    results = []
    debug_pages = []

    for page_num, raw_img in enumerate(raw_pages, start=1):
        preprocessed_img = preprocess(raw_img)
        pil_image = Image.fromarray(preprocessed_img[:, :, ::-1])
        ocr_result = _run_ocr(pil_image)
        full_text, avg_confidence = _build_text(ocr_result)

        tables, table_bboxes = extract_tables(preprocessed_img, ocr_result=ocr_result, page_index=page_num - 1)
        filtered_lines = mask_table_regions_from_text(ocr_result, table_bboxes)
        body_text = rebuild_body_text(filtered_lines)

        stamps = detect_circular_stamps(raw_img)
        page_output = build_page_output(page_num, full_text, avg_confidence, tables=tables)
        page_output["body_text_no_tables"] = body_text
        page_output["table_bboxes"] = [list(map(int, bbox)) for bbox in table_bboxes]

        if stamps:
            page_output["stamp_regions"] = [
                {"x": int(x), "y": int(y), "radius": int(r)}
                for x, y, r in stamps
            ]

        results.append(page_output)

        if include_debug:
            debug_pages.append({
                "page": page_num,
                "raw_image": raw_img,
                "preprocessed_image": preprocessed_img,
                "table_bboxes": [list(map(int, bbox)) for bbox in table_bboxes],
                "table_count": len(tables),
            })

    if include_debug:
        return results, debug_pages
    return results


def extract(ocr_output, doc_id=None):
    """
    KIE entry point: extract key information from OCR output.
    
    Args:
        ocr_output: List of page dicts from process_document()
        doc_id: Optional document identifier
    
    Returns:
        Structured document with extracted fields, doc type, etc.
    """
    if not doc_id:
        doc_id = f"doc_{id(ocr_output)}"

    full_text = "\n".join(p.get("raw_text", "") for p in ocr_output)
    doc_type = detect_doc_type(full_text)

    pages_fields = []
    for page in ocr_output:
        raw_text = page.get("raw_text", "")
        fields = extract_fields(raw_text)
        pages_fields.append(fields)

    return build_document_output(doc_id, ocr_output, pages_fields, doc_type)

print("✓ Main pipeline entry points (process_document, extract) loaded")

## 7. Evaluation Harness

Complete evaluation pipeline: run OCR and KIE on documents, calculate metrics, and generate reports.

In [ ]:
def evaluate(max_docs=None):
    """
    Main evaluation function: run OCR and KIE on documents and compute metrics.
    
    Args:
        max_docs: Limit number of documents to evaluate (for testing)
    
    Outputs:
        - full_ocr_eval.csv: Per-document OCR metrics
        - full_kie_eval.csv: Per-document × field KIE metrics
        - Summary tables printed to console
    """
    print("Loading pipeline…")
    gt_data = _load_gt()

    pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
    if max_docs:
        pdf_files = pdf_files[:max_docs]

    total = len(pdf_files)
    print(f"Found {total} document(s) to evaluate.\n")

    ocr_rows = []
    kie_rows = []

    ocr_acc = defaultdict(list)
    kie_acc = defaultdict(lambda: defaultdict(list))

    for idx, pdf_path in enumerate(pdf_files, 1):
        doc_name = pdf_path.name
        m = re.search(r"doc_(\d+)", doc_name)
        doc_num = m.group(1) if m else "???"
        gt_row = gt_data.get(doc_num, {})
        gt_full = _gt_full_text(gt_row) if gt_row else ""

        print(f"[{idx:3d}/{total}] {doc_name}", end=" … ", flush=True)

        t0 = time.perf_counter()
        try:
            ocr_out = process_document(str(pdf_path))
        except Exception as exc:
            print(f"OCR ERROR: {exc}")
            continue
        ocr_time = time.perf_counter() - t0

        ocr_text = " ".join(p.get("raw_text", "") for p in ocr_out)
        ocr_conf = sum(p.get("confidence", 0.0) for p in ocr_out) / max(len(ocr_out), 1)

        if gt_full:
            doc_cer = calc_cer(ocr_text, gt_full)
            doc_wer = calc_wer(ocr_text, gt_full)
            doc_lev = calc_levenshtein(ocr_text, gt_full)
            doc_tacc = calc_text_accuracy(ocr_text, gt_full)
        else:
            doc_cer = doc_wer = doc_lev = doc_tacc = float("nan")

        ocr_row = {
            "doc": doc_name,
            "doc_num": doc_num,
            "ocr_conf": round(ocr_conf, 4),
            "ocr_time_s": round(ocr_time, 3),
            "CER": _fmt(doc_cer),
            "WER": _fmt(doc_wer),
            "Levenshtein": _fmt(doc_lev),
            "Text_Accuracy": _fmt(doc_tacc),
        }
        ocr_rows.append(ocr_row)

        if gt_full:
            ocr_acc["CER"].append(doc_cer)
            ocr_acc["WER"].append(doc_wer)
            ocr_acc["Levenshtein"].append(doc_lev)
            ocr_acc["Text_Accuracy"].append(doc_tacc)
            ocr_acc["ocr_time_s"].append(ocr_time)

        t1 = time.perf_counter()
        try:
            kie_out = extract(ocr_out, doc_id=pdf_path.stem)
        except Exception as exc:
            print(f"KIE ERROR: {exc}")
            kie_out = {"pages": []}
        kie_time = time.perf_counter() - t1

        fields = kie_out["pages"][0]["fields"] if kie_out.get("pages") else {}

        for kie_field, gt_col in KIE_FIELDS.items():
            pred = fields.get(kie_field) or ""
            gold = gt_row.get(gt_col, "") or ""

            em = calc_exact_match(pred, gold, kie_field)
            prec, rec, f1 = calc_token_prf(pred, gold)
            facc = calc_field_accuracy(pred, gold)
            iou = calc_iou_text(pred, gold)

            kie_row = {
                "doc": doc_name,
                "doc_num": doc_num,
                "field": kie_field,
                "predicted": pred[:120],
                "ground_truth": gold[:120],
                "kie_time_s": round(kie_time, 3),
                "Exact_Match": _fmt(em),
                "Precision": _fmt(prec),
                "Recall": _fmt(rec),
                "F1": _fmt(f1),
                "Field_Accuracy": _fmt(facc),
                "IoU": _fmt(iou),
            }
            kie_rows.append(kie_row)

            if gold:
                kie_acc[kie_field]["Exact_Match"].append(em)
                kie_acc[kie_field]["Precision"].append(prec)
                kie_acc[kie_field]["Recall"].append(rec)
                kie_acc[kie_field]["F1"].append(f1)
                kie_acc[kie_field]["Field_Accuracy"].append(facc)
                kie_acc[kie_field]["IoU"].append(iou)

        cer_s = f"CER={doc_cer:.3f}" if doc_cer == doc_cer else "CER=n/a"
        wer_s = f"WER={doc_wer:.3f}" if doc_wer == doc_wer else "WER=n/a"
        tacc_s = f"Acc={doc_tacc:.1%}" if doc_tacc == doc_tacc else "Acc=n/a"
        avg_f1 = _safe_mean([
            _safe_mean(kie_acc[f]["F1"]) for f in KIE_FIELDS
            if kie_acc[f]["F1"]
        ])
        f1_s = f"KIE-F1={avg_f1:.3f}" if avg_f1 == avg_f1 else "KIE-F1=n/a"
        print(f"{cer_s}  {wer_s}  {tacc_s}  {f1_s}  OCR={ocr_time:.1f}s  KIE={kie_time:.2f}s")

    return ocr_rows, kie_rows, ocr_acc, kie_acc

print("✓ Evaluation harness loaded")

## 8. Generate and Export Results

Save evaluation metrics to CSV and display summary statistics.

In [ ]:
def export_results_and_report(ocr_rows, kie_rows, ocr_acc, kie_acc):
    """Export results to CSV and print summary statistics."""
    ocr_csv = ROOT / "full_ocr_eval.csv"
    kie_csv = ROOT / "full_kie_eval.csv"

    # Export OCR results
    with open(ocr_csv, "w", newline="", encoding="utf-8-sig") as f:
        if ocr_rows:
            w = csv.DictWriter(f, fieldnames=ocr_rows[0].keys())
            w.writeheader()
            w.writerows(ocr_rows)

    # Export KIE results
    with open(kie_csv, "w", newline="", encoding="utf-8-sig") as f:
        if kie_rows:
            w = csv.DictWriter(f, fieldnames=kie_rows[0].keys())
            w.writeheader()
            w.writerows(kie_rows)

    # Print OCR summary
    sep = "=" * 70
    print(f"\n{sep}")
    print("  OCR EVALUATION SUMMARY")
    print(sep)
    print(f"  {'Metric':<20} {'Mean':>10} {'Min':>10} {'Max':>10}")
    print("  " + "-" * 54)
    
    for metric in ["CER", "WER", "Levenshtein", "Text_Accuracy", "ocr_time_s"]:
        vals = ocr_acc[metric]
        if not vals:
            continue
        clean = [v for v in vals if v == v]
        if not clean:
            continue
        mean_v = sum(clean) / len(clean)
        print(f"  {metric:<20} {mean_v:>10.4f} {min(clean):>10.4f} {max(clean):>10.4f}")
    
    print(f"  Docs evaluated: {len(ocr_rows)}")
    print(f"  Saved → {ocr_csv}")

    # Print KIE summary
    print(f"\n{sep}")
    print("  KIE EVALUATION SUMMARY (mean over all documents)")
    print(sep)
    header = f"  {'Field':<14} {'N':>4} {'ExactMatch':>11} {'Precision':>10} {'Recall':>8} {'F1':>8} {'FieldAcc':>9} {'IoU':>8}"
    print(header)
    print("  " + "-" * 76)

    macro = defaultdict(list)
    for field in KIE_FIELDS:
        fa = kie_acc[field]
        n = len([v for v in fa["F1"] if v == v])
        em = _safe_mean(fa["Exact_Match"])
        pr = _safe_mean(fa["Precision"])
        rc = _safe_mean(fa["Recall"])
        f1 = _safe_mean(fa["F1"])
        ac = _safe_mean(fa["Field_Accuracy"])
        iu = _safe_mean(fa["IoU"])
        for k, v in [("EM", em), ("P", pr), ("R", rc), ("F1", f1), ("Acc", ac), ("IoU", iu)]:
            if v == v:
                macro[k].append(v)
        print(
            f"  {field:<14} {n:>4} "
            f"{_fmt(em):>11} {_fmt(pr):>10} {_fmt(rc):>8} "
            f"{_fmt(f1):>8} {_fmt(ac):>9} {_fmt(iu):>8}"
        )

    print("  " + "-" * 76)
    print(
        f"  {'Macro avg':<14} {'':>4} "
        f"{_fmt(_safe_mean(macro['EM'])):>11} "
        f"{_fmt(_safe_mean(macro['P'])):>10} "
        f"{_fmt(_safe_mean(macro['R'])):>8} "
        f"{_fmt(_safe_mean(macro['F1'])):>8} "
        f"{_fmt(_safe_mean(macro['Acc'])):>9} "
        f"{_fmt(_safe_mean(macro['IoU'])):>8}"
    )
    
    kie_time_all = [r['kie_time_s'] for r in kie_rows if r['field'] == 'body']
    if kie_time_all:
        print(f"\n  Mean KIE time/doc : {sum(kie_time_all)/len(kie_time_all):.3f}s")
    print(f"  Saved → {kie_csv}\n")

print("✓ Export and reporting functions loaded")

## 9. Example Usage and Testing

Run OCR, KIE, and evaluation on sample documents. Modify `max_docs` parameter to test on fewer documents.

In [ ]:
##############################################################################
# EXAMPLE: Run OCR + KIE on a single document
##############################################################################

# Find a PDF in documents folder
pdf_path = next(DOCS_DIR.glob("*.pdf"), None)

if pdf_path:
    print(f"Processing: {pdf_path.name}")
    print("=" * 70)
    
    # Step 1: OCR
    print("\n1️⃣  RUNNING OCR...")
    ocr_result = process_document(str(pdf_path))
    
    print(f"   ✓ Processed {len(ocr_result)} page(s)")
    for page in ocr_result:
        print(f"   • Page {page['page']}: confidence={page['confidence']:.2f}, "
              f"text_len={len(page['raw_text'])}, tables={len(page['tables'])}")
    
    # Step 2: KIE
    print("\n2️⃣  RUNNING KIE (KEY INFORMATION EXTRACTION)...")
    kie_result = extract(ocr_result, doc_id=pdf_path.stem)
    
    print(f"   ✓ Extracted from {kie_result['total_pages']} page(s)")
    print(f"   • Doc type: {kie_result['doc_type']} ({kie_result['doc_subtype']})")
    
    fields = kie_result['pages'][0]['fields']
    print(f"   • Extracted fields:")
    for field_name, field_value in fields.items():
        val_preview = str(field_value)[:50] if field_value else "(empty)"
        print(f"      - {field_name}: {val_preview}...")
    
    # Step 3: Show metrics example
    print("\n3️⃣  METRICS EXAMPLE (with dummy ground truth):")
    dummy_ocr = "Example OCR text output"
    dummy_ref = "Example reference text"
    cer = calc_cer(dummy_ocr, dummy_ref)
    wer = calc_wer(dummy_ocr, dummy_ref)
    print(f"   • CER: {cer:.4f}")
    print(f"   • WER: {wer:.4f}")

else:
    print("⚠️  No PDF files found in documents/ folder.")
    print("   Create documents/ folder with sample PDFs to test.")

print("\n" + "=" * 70)
print("To run FULL evaluation on all documents, use:")
print("  ocr_rows, kie_rows, ocr_acc, kie_acc = evaluate(max_docs=5)  # limit to 5 docs")
print("  export_results_and_report(ocr_rows, kie_rows, ocr_acc, kie_acc)")
print("=" * 70)

In [ ]:
# Run the full evaluation over all PDFs in documents/
ocr_rows, kie_rows, ocr_acc, kie_acc = evaluate(max_docs=None)
export_results_and_report(ocr_rows, kie_rows, ocr_acc, kie_acc)

## 10. Run Full Evaluation

Run evaluation across all PDFs in `documents/` and export the OCR/KIE summary CSV files.